# Practical 6: Transfer learning with pre-trained VGG16 (CIFAR-10)

**What we are doing.** **VGG16** was already trained on 1.2 million ImageNet photos, and its conv layers already know edges, textures and shapes. We keep them, **freeze** them, and train only a **new small classifier** for CIFAR-10's 10 classes. Then we **fine-tune**: unfreeze the last block and train it gently with a tiny learning rate (1e-5).

```
photo 32x32x3 → [VGG16 conv base, FROZEN] → 1x1x512 → Flatten → Dense 256 → Dropout → Dense 10 softmax
```
**Skeleton:** imports → load → preprocess → **model** → compile → fit → evaluate → plot/predict

> Use a **GPU** runtime. On a CPU this one is slow.

> **Run locally in Jupyter?** Install once: `%pip install tensorflow numpy pandas matplotlib scikit-learn`, then *Kernel → Restart Kernel*.
> **In Colab:** *Runtime → Change runtime type → T4 GPU*. Everything is preinstalled.

## Load and preprocess
- CIFAR-10: 32×32 **colour** photos in 10 classes. We use a 10k / 2k subset so it trains in minutes.
- `preprocess_input` prepares the pixels the way VGG16 was originally trained (colour order and mean subtraction), instead of ÷255.
- `x_test_raw` is an untouched copy used only for display.

In [ ]:
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import VGG16
from tensorflow.keras.applications.vgg16 import preprocess_input
import matplotlib.pyplot as plt

classes = ['airplane', 'automobile', 'bird', 'cat', 'deer',
           'dog', 'frog', 'horse', 'ship', 'truck']
(x_train, y_train), (x_test, y_test) = keras.datasets.cifar10.load_data()
x_train, y_train = x_train[:10000], y_train[:10000]   # a subset, so it trains in minutes
x_test, y_test = x_test[:2000], y_test[:2000]
x_test_raw = x_test.copy()                            # kept only for display
x_train = preprocess_input(x_train.astype('float32')) # VGG16's own preprocessing
x_test = preprocess_input(x_test.astype('float32'))

## a–c. Load VGG16, freeze it, add a custom classifier
- `weights='imagenet'`: the already-learned weights. `include_top=False`: drop the original 1000-class top.
- `base.trainable = False`: **freeze** about 14.7M weights. Only our new top (about 134k) trains.
- 32×32 shrinks through 5 poolings to **1×1×512**.

In [ ]:
# a. Load a CNN pre-trained on ImageNet, without its top classifier
base = VGG16(weights='imagenet', include_top=False, input_shape=(32, 32, 3))

# b. Freeze its weights
base.trainable = False

# c. Add a custom classifier with trainable layers
model = keras.Sequential([
    base,
    layers.Flatten(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(10, activation='softmax'),
])
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()   # note "Non-trainable params": the frozen VGG16

## d. Train only the classifier layers

In [ ]:
model.fit(x_train, y_train, epochs=5, batch_size=64, validation_split=0.1)

## e. Fine-tune the last block with a tiny learning rate
- Unfreeze everything, then re-freeze all **but the last 4** layers (`[:-4]` = all except the last 4). That leaves block 5 (3 conv + pool), about 7M weights.
- **Recompile** after changing `trainable`, or the change is ignored.
- `learning_rate=1e-5`: tiny nudges, so the pre-trained weights aren't wrecked.

In [ ]:
base.trainable = True
for layer in base.layers[:-4]:
    layer.trainable = False
model.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-5),
              loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.fit(x_train, y_train, epochs=3, batch_size=64, validation_split=0.1)

loss, acc = model.evaluate(x_test, y_test)
print("Test accuracy:", acc)

## Show predictions
Expect roughly **55–65%** on this small subset of tiny photos. The exam is about the method. `y_test[i][0]` because CIFAR labels look like `[[3], [8], ...]`.

In [ ]:
pred = model.predict(x_test[:10]).argmax(axis=1)
plt.figure(figsize=(14, 3))
for i in range(10):
    plt.subplot(1, 10, i + 1)
    plt.imshow(x_test_raw[i])
    plt.title(f"P:{classes[pred[i]]}\nA:{classes[y_test[i][0]]}", fontsize=8)
    plt.axis('off')
plt.show()

---
## Understand it: poke cells
Run B before and after stage e: unfreezing block 5 moves about 7 million weights to trainable.

In [ ]:
# A. What the frozen base outputs for one image
print(base(x_test[:1]).shape)   # (1, 1, 1, 512)

# B. Frozen vs trainable
print("trainable weights:", sum(w.numpy().size for w in model.trainable_weights))
print("frozen weights   :", sum(w.numpy().size for w in model.non_trainable_weights))

# C. Which layers got unfrozen
for l in base.layers[-5:]: print(l.name, l.trainable)

---
## Break it
In the fine-tune stage, use `optimizer='adam'` (default lr 0.001) instead of `1e-5`. Accuracy can collapse, because big updates wreck the pre-trained weights.

---
## Viva answers
**Q1. What does freezing do, and why freeze the lower layers?** A frozen layer's weights aren't updated. The lower layers detect general features (edges, colours, textures) that suit any image task, so we reuse them and train only the task-specific top.

**Q2. Why `include_top=False`, and what replaces it?** The original top classifies 1000 ImageNet classes. We drop it and add Flatten → Dense 256 → Dropout → Dense 10 softmax.

**Q3. Why a much smaller learning rate for fine-tuning?** The unfrozen weights are already good. Large updates would destroy them, so we nudge gently (1e-5).

**Worth saying:** this is image *classification* with transfer learning. True object detection also predicts bounding boxes (YOLO, Faster R-CNN).

> **Classmate's code, 4 bugs:** (1) the typo `layer.trainabler = False` means nothing gets frozen; (2) `layers[:-4]` was set trainable, which unfreezes the *lower* layers; (3) Adam's default lr during fine-tuning dropped accuracy from 57% to 26%; (4) the "test" batch came from the training folder.